# Lab: ajustando uma SVM em retornos de radar

Notebook de aula da página: https://insper.github.io/ML/pt/2026.2/classes/svm/

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e olhar

**Preveja.** Antes de rodar: para uma SVM, o que importa mais saber sobre as 34 colunas — a média delas, a faixa de valores ou quantas são categóricas? Por quê?

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

data = fetch_openml(data_id=59, as_frame=True)       # Ionosphere (UCI)
X = data.data                                        # 34 atributos numéricos
y = (data.target == 'g').astype(int)                 # 1 = retorno de radar "bom", 0 = "ruim"
print(X.shape)
print(y.value_counts())
print(f"values from {X.min().min():.0f} to {X.max().max():.0f} · "
      f"constant columns: {list(X.columns[X.std() == 0])}")

## Passo 2 — trancar o conjunto de teste

**Preveja.** Com 88 linhas de teste, quanto vale uma única linha em pontos de acurácia? Isso muda o quanto você deve confiar na terceira casa decimal do número final?

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(f"share of 'good': train {y_train.mean():.3f} · test {y_test.mean():.3f}")

## Passo 3 — um baseline e os folds que todos compartilham

**Preveja.** Responder sempre "bom" acerta quantos por cento? E qual vai ser a ROC AUC desse baseline?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def cv_summary(model):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=['accuracy', 'roc_auc'])
    return {metric: f"{r['test_' + metric].mean():.3f} ± {r['test_' + metric].std():.3f}"
            for metric in ['accuracy', 'roc_auc']}


print(cv_summary(DummyClassifier(strategy='most_frequent')))

## Passo 4 — três modelos que você conhece, nos mesmos folds

**Preveja.** A regressão logística e a SVM linear traçam ambas um hiperplano nos mesmos 34 atributos padronizados e só discordam na perda. Você espera uma diferença grande entre elas ou menor que o desvio entre folds?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

models = {
    'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'k-NN (k = 5)': make_pipeline(StandardScaler(), KNeighborsClassifier()),
    'linear SVM (C = 1)': make_pipeline(StandardScaler(), SVC(kernel='linear')),
}
print(pd.DataFrame({name: cv_summary(m) for name, m in models.items()}).T)

**Simulação: só os vetores de suporte importam.** A seção 2 afirmou que apagar um ponto que não é vetor de suporte não muda nada. Teste isso nos dados do lab: padronize **uma vez** (para isolar o efeito, já que um scaler reajustado mudaria tudo), ajuste uma SVM linear, descubra quais linhas são vetores de suporte e refaça o ajuste sem uma linha de cada tipo. O que acontece com $\lVert w \rVert$?

In [ ]:
# sua simulação aqui

**Simulação: o C muda a largura da rua.** Varra o `C` da SVM linear de 0,01 a 100 e meça, para cada valor, a largura da rua $2/\lVert w \rVert$, quantos vetores de suporte sobram, a acurácia de treino e a de validação cruzada. Onde a CV é máxima e o que acontece com ela depois disso?

In [ ]:
# sua simulação aqui

**Para fechar a parte fundamental.** Em uma frase cada: (1) por que a acurácia de treino não consegue escolher entre duas retas que separam os dados? (2) O que muda na solução se você apagar um ponto que está além da margem? (3) O que um $C$ pequeno faz com a largura da rua e com o número de vetores de suporte?

## Sua vez (fundamental)

Para casa:

1. **A margem à mão.** Pegue os seis pontos da seção 2 e mova o ponto D de (2, 2) para (2,5; 2,5). Refaça a conta à mão (a fronteira continua perpendicular à diagonal) e confira com `SVC(kernel='linear', C=1e6)`. A rua ficou mais larga ou mais estreita?
2. **Hinge contra log loss.** Para $m$ de −3 a 3, faça o gráfico de `np.maximum(0, 1 - m)` e de `np.log1p(np.exp(-m))`. Em que ponto elas se cruzam? Qual das duas pune mais um erro grosseiro e qual continua "escutando" um ponto bem classificado?
3. **Quantos vetores de suporte?** No lab, refaça a simulação do $C$ com `kernel='linear'` em `load_breast_cancer`. A relação entre $C$ e o número de vetores de suporte é a mesma? E a curva da CV tem um arco parecido?

## Passo 5 — ligar o kernel RBF

**Preveja.** A única mudança é `kernel='rbf'`, sem ajuste nenhum. O ganho sobre a SVM linear vai ser maior ou menor que o desvio entre folds do passo 4 (cerca de 0,02)?

In [ ]:
rbf = make_pipeline(StandardScaler(), SVC(kernel='rbf'))       # padrões: C = 1, gamma = 'scale'
print(cv_summary(rbf))

Z = StandardScaler().fit_transform(X_train)
print(f"gamma='scale' means gamma = 1 / (n_features · variance) = {1 / (Z.shape[1] * Z.var()):.4f}")

## Passo 6 — ajustar C e γ juntos

**Preveja.** Numa grade de 25 pares $(C, \gamma)$, você espera que o melhor valor de $C$ dependa do $\gamma$ escolhido? E qual canto da tabela deve ser o pior: $\gamma$ muito pequeno ou muito grande?

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV

param_grid = {'svc__C': [0.1, 1, 10, 100, 1000],
              'svc__gamma': [0.0001, 0.001, 0.01, 0.1, 1]}
# selecao do modelo
grid = GridSearchCV(rbf, param_grid, cv=cv, scoring='accuracy', return_train_score=True)
grid.fit(X_train, y_train)

res = pd.DataFrame(grid.cv_results_)
table = res.pivot(index='param_svc__C', columns='param_svc__gamma', values='mean_test_score')
print(table.round(3))
best_std = res.loc[grid.best_index_, 'std_test_score']
print(f"best: {grid.best_params_} · accuracy {grid.best_score_:.3f} ± {best_std:.3f}")
ties = res[res['mean_test_score'] >= grid.best_score_ - best_std]
print(f"{len(ties)} of {len(res)} pairs are within one std of the best")
corner = (res['param_svc__C'] == 1) & (res['param_svc__gamma'] == 1)
print(f"training-fold accuracy · best pair {res.loc[grid.best_index_, 'mean_train_score']:.3f} · "
      f"C = 1, gamma = 1: {res.loc[corner, 'mean_train_score'].item():.3f}")

plt.imshow(table, cmap='Blues', origin='lower')
plt.xticks(range(5), table.columns); plt.yticks(range(5), table.index)
plt.xlabel('gamma'); plt.ylabel('C'); plt.colorbar(label='CV accuracy')
plt.show()

**Simulação: o que o γ faz, visto de dentro.** Fixe `C = 1` e varra o $\gamma$ de 0,001 a 1. Para cada valor, conte os vetores de suporte e compare a acurácia de treino com a de validação cruzada. O vão entre as duas conta a história do sobreajuste melhor que qualquer um dos números sozinho.

In [ ]:
# sua simulação aqui

## Passo 7 — por dentro do vencedor: os vetores de suporte

**Preveja.** O vencedor tem $\gamma = 0{,}1$, razoavelmente local, em 34 dimensões. Você espera que ele guarde uns 10% das linhas de treino como vetores de suporte ou bem mais?

In [ ]:
best = grid.best_estimator_                     # reajustado em todo o X_train
svc = best[-1]
print(f"support vectors (bad, good): {svc.n_support_} · {svc.n_support_.sum()} of {len(X_train)} rows")

alpha = np.abs(svc.dual_coef_[0])
print(f"on the margin (0 < alpha < C): {(alpha < svc.C - 1e-6).sum()} · "
      f"inside it or misclassified (alpha = C): {(alpha >= svc.C - 1e-6).sum()}")

m = np.where(y_train == 1, 1, -1) * best.decision_function(X_train)   # m = y · f(x)
print(f"training rows with m < 0 (misclassified): {(m < 0).sum()} · "
      f"training accuracy {best.score(X_train, y_train):.3f}")

**Simulação: escalonar num conjunto que já está normalizado.** O passo 1 notou que todos os atributos já estão entre −1 e 1. Então o `StandardScaler` ainda faz diferença aqui? Meça a acurácia de CV do vencedor com e sem ele e olhe antes a faixa de desvios-padrão das colunas.

In [ ]:
# sua simulação aqui

## Passo 8 — a SVM é mesmo melhor? Rivais ajustados, mesmos folds

**Preveja.** Agora os rivais também são ajustados. A vantagem da SVM RBF vai sobreviver a isso? E por que o desvio das *diferenças* por fold deve ser menor que o desvio de cada modelo?

In [ ]:
rivals = {
    'logistic regression': GridSearchCV(models['logistic regression'],
                                        {'logisticregression__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
                                        cv=cv),
    'k-NN': GridSearchCV(models['k-NN (k = 5)'],
                         {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 15, 25]}, cv=cv),
}


def fold_scores(g):                              # os 5 escores por fold do vencedor de uma grade
    return [g.cv_results_[f'split{k}_test_score'][g.best_index_] for k in range(5)]


folds = pd.DataFrame({'RBF SVM': fold_scores(grid)})
for name, g in rivals.items():
    g.fit(X_train, y_train)
    print(f"{name}: {g.best_params_}")
    folds[name] = fold_scores(g)
folds['SVM - LR'] = folds['RBF SVM'] - folds['logistic regression']
folds['SVM - k-NN'] = folds['RBF SVM'] - folds['k-NN']
print(folds.round(3))
print(pd.DataFrame({'mean': folds.mean(), 'std': folds.std(ddof=0)}).T.round(3))

## Passo 9 — o único teste

**Preveja.** A CV disse 0,954 ± 0,019. Com 88 linhas de teste, quanto um único erro vale? Um resultado alguns pontos abaixo da CV seria motivo para voltar e reajustar?

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pred = best.predict(X_test)
print(confusion_matrix(y_test, pred, labels=[1, 0]))
print(classification_report(y_test, pred, target_names=['bad', 'good'], digits=3))
print(f"test ROC AUC (from decision_function): {roc_auc_score(y_test, best.decision_function(X_test)):.3f}")

**Para fechar a aula.** Em uma frase cada: (1) o que o truque do kernel evita calcular? (2) Por que $C$ e $\gamma$ precisam ser buscados juntos? (3) Quando uma SVM com kernel deixa de ser prática e o que usar no lugar?

## Sua vez (avançado)

Para casa, com mais tempo:

1. **Kernel polinomial.** Troque o RBF por `SVC(kernel='poly', coef0=1)` e ajuste `degree` (2, 3, 4) e `C` com o mesmo `cv`. Ele chega a menos de um desvio-padrão do vencedor RBF? Quantos vetores de suporte ele guarda?
2. **CV aninhada.** Rode `cross_val_score(grid, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=7))`. Esse escore avalia o *procedimento de ajuste inteiro* em folds que o ajuste nunca viu. Compare com `grid.best_score_` e com a acurácia de teste do passo 9: de qual dos dois ele fica mais perto?
3. **Probabilidades.** Monte `CalibratedClassifierCV(grid.best_estimator_, method='sigmoid', ensemble=False, cv=cv)` e compare a ROC AUC dele em validação cruzada (a `cv_summary` funciona com ele) com a do próprio vencedor (`cv_summary(grid.best_estimator_)`). Por que as duas ficam tão próximas? Depois use `cross_val_predict` nos dois para contar as linhas de treino cuja classe prevista muda.
4. **Multiclasse.** No `load_digits`, ajuste `C` e `gamma` da `SVC` com um `GridSearchCV` e um `cv` estratificado de 5 folds. Quantos classificadores de pares o vencedor treina e quantos vetores de suporte ele guarda para o dígito 8 comparado ao dígito 0? Quais dígitos são os mais difíceis, segundo a matriz de confusão das previsões fora do fold?